# 3.4 分布式 SpMV 实现

## 本节学习目标

- 串联 MPI_Bcast、local SpMV 和 MPI_Gatherv
- 理解最大时间归约

## 环境检查

直接检查本节需要的运行环境；若检查失败，请先在对应 CPU/NPU 节点加载课程要求的工具链。


In [ ]:
import shutil, subprocess
for tool in ("cmake", "mpicxx", "mpirun"):
    path = shutil.which(tool)
    if path is None:
        raise RuntimeError(f"缺少必需工具：{tool}")
    print(f"{tool}: {path}")


## 每轮数据流

rank 0 持有完整 x；`MPI_Bcast` 广播后，每个 rank 对本地 CSR 行执行同一串行内核；`MPI_Gatherv` 根据 row_counts/displacements 恢复全局 y。

## 计时口径

每轮先 Barrier，再分别统计 broadcast、compute、gather。最后以 `MPI_Reduce(..., MPI_MAX)` 取最慢 rank，符合分布式作业由临界路径决定的事实。

## 查看核心循环

从当前章节目录执行下面的 Cell，并对照随后给出的检查点阅读输出。

## 构建与快速运行

从当前章节目录执行下面的 Cell，并对照随后给出的检查点阅读输出。

In [ ]:
!cd src/mpi_spmv && bash scripts/build.sh
!cd src/mpi_spmv && MPI_PROCESSES=2 bash scripts/run_mpi.sh --matrix U1 --warmup 1 --repeat 3

## 预期现象与结果分析

rank 0 应输出 CPU baseline、MPI 端到端时间、分阶段耗时、负载均衡和 PASS/FAIL。进程数不能超过调度器分配资源。

## 课后实践

解释为何分阶段最大值不能简单相加还原端到端最大值。

参考答案见 `answer/03.04_answer.md`。